# 01 — Data Loading and Cleaning

Builds `df_development` (2000-2021, used for modelling) and `df_real_world` (2022-2023, unlabeled, kept for reference only) from three sources:

1. **IHME** — mental health disorder prevalence (local CSV, already filtered to EU countries at extraction time)
2. **World Bank API** — socioeconomic indicators
3. **WHO API** — suicide rate (target variable)

EDA starts in `02_eda.ipynb`, once this notebook has produced `data/processed/df_development.parquet`.

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd

from src.config import (
    EU_COUNTRIES_ISO,
    WORLD_BANK_INDICATORS,
    WHO_SUICIDE_INDICATOR,
)
from src.data_loading import (
    load_ihme_data,
    fetch_worldbank_indicators,
    fetch_who_suicide_rates,
    build_master_dataset,
    impute_missing_values,
)

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
np.random.seed(42)

## Step 1 — IHME mental health prevalence (base dataset)

Read from a local CSV. Replace the path below with the location of your IHME export if it differs.

In [ ]:
IHME_CSV_PATH = "../data/raw/IHME-GBD_2023_DATA-b62eec84-1.csv"

df_base = load_ihme_data(IHME_CSV_PATH, min_year=2000)
print(f"IHME base dataset: {df_base.shape[0]} rows, {df_base.shape[1]} columns")
display(df_base.head())

## Step 2 — World Bank socioeconomic indicators (API)

One request per indicator (the World Bank API does not support multi-indicator queries), merged onto `df_base` by `Code` + `Year`.

In [ ]:
df_features = fetch_worldbank_indicators(
    df_base,
    eu_countries_iso=EU_COUNTRIES_ISO,
    indicators=WORLD_BANK_INDICATORS,
    region="ALL",
    date_range="2000:2026",
)
print(
    f"\nFeature dataset after World Bank merge: {df_features.shape[0]} rows, {df_features.shape[1]} columns"
)
display(df_features.head())

## Step 3 — WHO suicide rate (target variable, API)

Filtered to both sexes (`SEX_BTSX`) and all age groups (`AGEGROUP_YEARSALL`), then merged onto `df_features`.

In [ ]:
df_who = fetch_who_suicide_rates(indicator=WHO_SUICIDE_INDICATOR)

df_development, df_real_world = build_master_dataset(
    df_features, df_who, development_cutoff_year=2021
)

print(
    f"df_development: {df_development.shape[0]} rows (2000-2021, labeled — used for modelling)"
)
print(
    f"df_real_world:  {df_real_world.shape[0]} rows (2022-2023, unlabeled — reference only)"
)
display(df_development.head())

## Descriptive check and missing values

In [ ]:
print("Master dataset summary statistics")
display(df_development.describe().T)

print("\nMissing values per column:")
print(df_development.isnull().sum())

In [ ]:
# Inspect rows with missing Gini index or Physicians per 100,000,
# to decide on an imputation method
missing_mask = (
    df_development["Gini index"].isna() | df_development["Physicians per 100000"].isna()
)
display(df_development[missing_mask])

## Missing value imputation

Linear interpolation within each country, chosen because the missing values occur in consecutive year runs rather than scattered individual years.

In [ ]:
features_with_nan = ["Gini index", "Physicians per 100000"]

df_development = impute_missing_values(df_development, features_with_nan)

print("Remaining NaNs after interpolation:")
print(df_development[features_with_nan].isnull().sum())
display(df_development.head())

## Save processed data

Saved so downstream notebooks (`02_eda.ipynb` onward) can load directly without re-running the API calls above.

In [ ]:
df_development.to_parquet("../data/processed/df_development.parquet")
df_real_world.to_parquet("../data/processed/df_real_world.parquet")

print("Saved: data/processed/df_development.parquet")
print("Saved: data/processed/df_real_world.parquet")